# 02 — Load Static GTFS
Downloads `SEQ_GTFS.zip` (if not already present for today), extracts CSVs,
loads into DataFrames, and saves as Parquet for fast EDA loading.

In [ ]:
# Cell 1 — Download SEQ_GTFS.zip if not already present for today
import requests
import yaml
import zipfile
from datetime import date
from pathlib import Path

REPO_ROOT = Path.cwd().parent
STATIC_DIR = REPO_ROOT / 'source_files' / 'gtfs_static'

with open(REPO_ROOT / 'config' / 'feeds.yaml') as f:
    config = yaml.safe_load(f)

STATIC_URL = config['gtfs_static']['seq']
today = date.today().strftime('%Y-%m-%d')
today_dir = STATIC_DIR / today

if today_dir.exists() and any(today_dir.iterdir()):
    print(f'Static GTFS already present for {today}: {today_dir}')
else:
    print(f'Downloading SEQ_GTFS.zip for {today} ...')
    zip_path = STATIC_DIR / 'SEQ_GTFS.zip'
    resp = requests.get(STATIC_URL, timeout=120, stream=True)
    resp.raise_for_status()
    with open(zip_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    today_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(today_dir)
    zip_path.unlink()
    print(f'Extracted to {today_dir.relative_to(REPO_ROOT)}')

GTFS_DIR = today_dir
print(f'\nLoading from: {GTFS_DIR.relative_to(REPO_ROOT)}')
print('Files:', [f.name for f in GTFS_DIR.glob('*.txt')])

In [ ]:
# Cell 2 — Load key GTFS files into DataFrames
import pandas as pd

routes = pd.read_csv(GTFS_DIR / 'routes.txt', dtype=str,
                     usecols=['route_id', 'route_short_name', 'route_type'])

trips = pd.read_csv(GTFS_DIR / 'trips.txt', dtype=str,
                    usecols=['trip_id', 'route_id', 'direction_id', 'shape_id'])

stop_times = pd.read_csv(GTFS_DIR / 'stop_times.txt', dtype=str,
                          usecols=['trip_id', 'arrival_time', 'departure_time',
                                   'stop_id', 'stop_sequence'])

stops = pd.read_csv(GTFS_DIR / 'stops.txt', dtype=str,
                    usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
stops['stop_lat'] = stops['stop_lat'].astype(float)
stops['stop_lon'] = stops['stop_lon'].astype(float)

cal_path = GTFS_DIR / 'calendar.txt'
cal_dates_path = GTFS_DIR / 'calendar_dates.txt'
calendar = pd.read_csv(cal_path, dtype=str) if cal_path.exists() else pd.DataFrame()
calendar_dates = pd.read_csv(cal_dates_path, dtype=str) if cal_dates_path.exists() else pd.DataFrame()

for name, df in [('routes', routes), ('trips', trips), ('stop_times', stop_times),
                  ('stops', stops), ('calendar', calendar), ('calendar_dates', calendar_dates)]:
    print(f'{name:>15}: {df.shape[0]:>8,} rows × {df.shape[1]} cols')

In [ ]:
# Cell 3 — Show shape and sample rows for each DataFrame
print('=== routes ===');    display(routes.head(3))
print('=== trips ===');     display(trips.head(3))
print('=== stop_times ===');display(stop_times.head(3))
print('=== stops ===');     display(stops.head(3))
if not calendar.empty:
    print('=== calendar ==='); display(calendar.head(3))
if not calendar_dates.empty:
    print('=== calendar_dates ==='); display(calendar_dates.head(3))

In [ ]:
# Cell 4 — Key join: stop_times → trips → routes
# Answers: which route + direction serves stop X?

EXAMPLE_STOP = stops['stop_id'].iloc[0]

stop_routes = (
    stop_times[stop_times['stop_id'] == EXAMPLE_STOP]
    .merge(trips[['trip_id', 'route_id', 'direction_id']], on='trip_id', how='left')
    .merge(routes[['route_id', 'route_short_name', 'route_type']], on='route_id', how='left')
    [['stop_id', 'route_id', 'route_short_name', 'route_type', 'direction_id',
      'arrival_time', 'stop_sequence']]
    .drop_duplicates(subset=['route_id', 'direction_id'])
    .sort_values('route_short_name')
)

stop_name = stops.loc[stops['stop_id'] == EXAMPLE_STOP, 'stop_name'].iloc[0]
print(f'Routes serving stop {EXAMPLE_STOP} ({stop_name}):')
display(stop_routes)

In [ ]:
# Cell 5 — Save DataFrames as Parquet for fast loading in EDA notebook
import pyarrow  # noqa — ensures pyarrow engine available

parquet_dir = STATIC_DIR / today
parquet_dir.mkdir(parents=True, exist_ok=True)

to_save = [
    ('routes', routes),
    ('trips', trips),
    ('stop_times', stop_times),
    ('stops', stops),
]
if not calendar.empty:
    to_save.append(('calendar', calendar))
if not calendar_dates.empty:
    to_save.append(('calendar_dates', calendar_dates))

for name, df in to_save:
    path = parquet_dir / f'{name}.parquet'
    df.to_parquet(path, index=False)
    print(f'Saved {name}.parquet ({len(df):,} rows) → {path.relative_to(REPO_ROOT)}')

print('\nNext step: run notebooks/03_load_performance_data.ipynb')